# Processing MESACLIP data - atmosphere daily, single ensembles

* **Description**: Reads in MESACLIP atmospheric data
* **Creator**: Alice DuVivier
* **Date**: December 2025

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from glob import glob
from datetime import datetime
import xesmf as xe
import matplotlib.pyplot as plt
import scipy.sparse as sps

import dask
from distributed import Client
from ncar_jobqueue import NCARCluster

## Spin up DASK cluster

In [2]:
# Create our NCAR Cluster - which uses PBSCluster under the hood
cluster = NCARCluster(walltime='12:00:00', cores=1, processes=1, memory='100 GB', 
                      resource_spec='select=1:ncpus=1:mem=100GB',interface='ext')

# Spin up 32 workers
cluster.scale(32)

# Assign the cluster to our Client
client = Client(cluster)

/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/dask_jobqueue/core.py:274: FutureWarning: env_extra has been renamed to job_script_prologue. You are still using it (even if only set to []; please also check config files). If you did not set job_script_prologue yet, env_extra will be respected for now, but it will be removed in a future release. If you already set job_script_prologue, env_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/con

In [3]:
client
# cluster.close()

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/duvivier/PBS/proxy/37015/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/duvivier/PBS/proxy/37015/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.174:35669,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/duvivier/PBS/proxy/37015/status,Total threads: 0
Started: Just now,Total memory: 0 B


## Set MESACLIP ensemble information

In [4]:
# RCP 8.5
path_base = '/gdex/data/d651009/'
case_base = 'b.e13.BRCP85C5.ne120_t12.cesm-ihesp-'
# list 10 members of MESACLIP - last one is the first and named differently
ens_mems = ['sehires38-2006-2100.001', 'hires1.0.30-2006-2100.002', 
            'hires1.0.31-2006-2100.003', 'hires1.0.44-2006-2100.004', 
            'hires1.0.44-2006-2100.005', 'hires1.0.46-2006-2100.006', 
            'hires1.0.46-2006-2100.007', 'hires1.0.46-2006-2100.008',
            'hires1.0.46-2006-2100.009', 'hires1.0.46-2006-2100.010']

nens = len(ens_mems)

In [5]:
# set the yearly data since we have to load them separately
# 2006 - 2100
years_all = np.arange(2006,2101,1)

nyears = len(years_all)

## Remapping files for a regular grid

In [6]:
#wgtfile = '/glade/work/fredc/Regrid_Maps/ne120np4_pentagons_to_latlon_0.25x0.25_0E_bilinear_20210903.nc'
wgtfile = '/glade/work/fredc/Regrid_Maps/ne120np4_pentagons_to_latlon_0.25x0.25_0E_conserve_20200309.nc'
dsw = xr.open_dataset(wgtfile)

In [7]:
def remap_camse(ds, dsw, varlst=[]):
    dso = xr.full_like(ds.drop_dims('ncol'), np.nan)
    lonb = dsw.xc_b.values.reshape([dsw.dst_grid_dims[1].values, dsw.dst_grid_dims[0].values])
    latb = dsw.yc_b.values.reshape([dsw.dst_grid_dims[1].values, dsw.dst_grid_dims[0].values])
    weights = sps.coo_matrix((dsw.S, (dsw.row-1, dsw.col-1)), shape=[dsw.sizes['n_b'], dsw.sizes['n_a']])
    if not varlst:
        for varname in list(ds):
            if 'ncol' in(ds[varname].dims):
                varlst.append(varname)
        if 'lon' in varlst: varlst.remove('lon')
        if 'lat' in varlst: varlst.remove('lat')
        if 'area' in varlst: varlst.remove('area')
    for varname in varlst:
        shape = ds[varname].shape
        invar_flat = ds[varname].values.reshape(-1, shape[-1])
        remapped_flat = weights.dot(invar_flat.T).T
        remapped = remapped_flat.reshape([*shape[0:-1], dsw.dst_grid_dims[1].values, dsw.dst_grid_dims[0].values])
        dimlst = list(ds[varname].dims[0:-1])
        dims={}
        coords={}
        for it in dimlst:
            dims[it] = dso.sizes[it]
            coords[it] = dso.coords[it]
        dims['lat'] = int(dsw.dst_grid_dims[1])
        dims['lon'] = int(dsw.dst_grid_dims[0])
        coords['lat'] = latb[:,0]
        coords['lon'] = lonb[0,:]
        remapped = xr.DataArray(remapped, coords=coords, dims=dims, attrs=ds[varname].attrs)
        dso = xr.merge([dso, remapped.to_dataset(name=varname)])
    varlst.clear() #not sure why this is needed but varlst was not cleared and it created issues with successive call to remap_camse
    return dso

## Load in ATMOSPHERIC Data
- No 6h data in ens 001

In [8]:
# Choose variable to load
# 'UBOT', 'VBOT', 'PSL'
var_in = 'VBOT'

In [9]:
# set output directory
path_out = '/glade/campaign/cgd/ppc/duvivier/arctic_actionable/DATA/mesaclip_all_arctic/daily/'+var_in+'/'

In [10]:
# loop through each ensemble and each set of years

# loop through ensembles
for n in np.arange(0,nens,1)[1:11]:
    # loop through year groupings
    for y in np.arange(0,nyears,1):
        ens_in = ens_mems[n]
        yr_pre = years_all[y]
        print('loading ensemble: '+ens_in+' for year '+str(yr_pre))
        # set case
        case = case_base + ens_in
        # set data directory
        path = path_base+case+'/atm/proc/tseries/hour_6'
        # get list of files
        files = sorted(glob(f'{path}/{case}.cam.h4.{var_in}.{yr_pre}??????-??????????.nc'))
        # load file
        ds_tmp = xr.open_mfdataset(files, data_vars="minimal", coords='minimal', compat="override", parallel=True, 
                           concat_dim="time", combine='nested', decode_times=True)
        # take an average over the time bounds to get the right time dimension
        ds_tmp["time"] =  ds_tmp.time_bnds.compute().mean(dim="nbnd")
        # take 6h averages and get daily averages
        ds_tmp = ds_tmp.resample(time='D').mean()
        # remap to a regular grid
        ds = remap_camse(ds_tmp,dsw)
        # crop to just the Alaskan Arctic Region: 50-80N Lat, 170-230 Lon
        ds_sub = ds.isel(lat=slice(560,720)).isel(lon=slice(680,920))

        # save and export data
        # set output file name
        nn = n+1
        if nn < 10:
            ensnum = '00'+str(nn)
        elif nn >= 10:
            ensnum = '010'

        file_out = 'MESACLIP_RCP8.5_'+ensnum+'_'+str(yr_pre)+'_'+var_in+'_alaska_daily.nc'
        fout = path_out + file_out
        #print(fout)

        # make output array
        ds_out = ds_sub
        # change the attributes
        ds_out.attrs['author'] = 'Alice DuVivier'
        ds_out.attrs['date_processed'] = datetime.now().strftime('%Y-%m-%d')
        ds_out.attrs['contents'] = f'Subset of RCP8.5 MESACLIP Data'
        # calculate the size of the dataset in GB
        size_gb = ds_out.nbytes / (1024**3)
        #print(f"The dataset is approximately {size_gb:.2f} GB.")

        # Export the dataset to NetCDF with all attributes and coordinates
        print('Writing post processed file')
        ds_out.to_netcdf(fout)

loading ensemble: hires1.0.30-2006-2100.002 for year 2006


/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/numpy/core/numeric.py:407: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


Writing post processed file
loading ensemble: hires1.0.30-2006-2100.002 for year 2007


/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/numpy/core/numeric.py:407: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


Writing post processed file
loading ensemble: hires1.0.30-2006-2100.002 for year 2008


...

Writing post processed file
loading ensemble: hires1.0.46-2006-2100.010 for year 2099


/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/numpy/core/numeric.py:407: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


Writing post processed file
loading ensemble: hires1.0.46-2006-2100.010 for year 2100


/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/numpy/core/numeric.py:407: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


Writing post processed file


In [11]:
print("Done!")
cluster.close()

Done!


/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/dask_jobqueue/core.py:274: FutureWarning: env_extra has been renamed to job_script_prologue. You are still using it (even if only set to []; please also check config files). If you did not set job_script_prologue yet, env_extra will be respected for now, but it will be removed in a future release. If you already set job_script_prologue, env_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
